In [1]:
import numpy as np
import pandas as pd

from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import csv

In [2]:
class GridWorld:
    def __init__(self):
        self.gridworld = [[0] * 9 for i in range(6)]

        self.gridworld[1][2] = 2
        self.gridworld[2][2] = 2
        self.gridworld[3][2] = 2

        self.gridworld[0][7] = 2
        self.gridworld[1][7] = 2
        self.gridworld[2][7] = 2

        self.gridworld[4][5] = 2

        self.gridworld[2][0] = 1

        self.agent_location_x = 2
        self.agent_location_y = 0

    def action_space(self):
        return [0, 1, 2, 3]

    def step(self, a):
        if a == 0:
            self.agent_location_y_temp = max(self.agent_location_y - 1, 0)

            if self.gridworld[self.agent_location_x][self.agent_location_y_temp] == 2:
                return self.agent_location_x, self.agent_location_y

            self.gridworld[self.agent_location_x][self.agent_location_y] = 0
            self.agent_location_y = self.agent_location_y_temp
            self.gridworld[self.agent_location_x][self.agent_location_y] = 1

        if a == 1:
            self.agent_location_x_temp = min(self.agent_location_x + 1, 5)

            if self.gridworld[self.agent_location_x_temp][self.agent_location_y] == 2:
                return self.agent_location_x, self.agent_location_y

            self.gridworld[self.agent_location_x][self.agent_location_y] = 0
            self.agent_location_x = self.agent_location_x_temp
            self.gridworld[self.agent_location_x][self.agent_location_y] = 1

        if a == 2:
            self.agent_location_y_temp = min(self.agent_location_y + 1, 8)

            if self.gridworld[self.agent_location_x][self.agent_location_y_temp] == 2:
                return self.agent_location_x, self.agent_location_y

            self.gridworld[self.agent_location_x][self.agent_location_y] = 0
            self.agent_location_y = self.agent_location_y_temp
            self.gridworld[self.agent_location_x][self.agent_location_y] = 1

        if a == 3:
            self.agent_location_x_temp = max(self.agent_location_x - 1, 0)

            if self.gridworld[self.agent_location_x_temp][self.agent_location_y] == 2:
                return self.agent_location_x, self.agent_location_y

            self.gridworld[self.agent_location_x][self.agent_location_y] = 0
            self.agent_location_x = self.agent_location_x_temp
            self.gridworld[self.agent_location_x][self.agent_location_y] = 1

        return self.agent_location_x, self.agent_location_y

    def reset(self):
        self.gridworld[1][2] = 2
        self.gridworld[2][2] = 2
        self.gridworld[3][2] = 2

        self.gridworld[0][7] = 2
        self.gridworld[1][7] = 2
        self.gridworld[2][7] = 2

        self.gridworld[4][5] = 2

        self.gridworld[2][0] = 1

        self.agent_location_x = 2
        self.agent_location_y = 0

In [3]:
env = GridWorld()

In [4]:
def one_hot_encoding(action):
    if action == 0:
        return [1, 0, 0, 0]
    elif action == 1:
        return [0, 1, 0, 0]
    elif action == 2:
        return [0, 0, 1, 0]
    elif action == 3:
        return [0, 0, 0, 1]

Gotta put in the inverted model sampling here.

In [5]:
def do_n_samples(n, inverse=False):

    data_X = []
    data_Y = []

    for i in range(n):
        previous_state = env.agent_location_x, env.agent_location_y

        action = np.random.randint(4)
        next_state = env.step(action)


        if inverse == True:
            input = list(previous_state) + list(next_state)

            output = one_hot_encoding(action)
        else:

          input = list(previous_state) + one_hot_encoding(action)

          output = list(next_state)

        data_X.append(input)

        data_Y.append(output)

    return data_X, data_Y

Writing out the program for intrinsic motivation.

Essentially, there are a few things here:

1. We need to compute an inverse dynamics model which is basically computes the action taken given a particular starting state and a particular ending state
2. We need to compute a model that would predict an particular ending state given a starting state and an action.

It need to be incredibly good at computing these two objectives. A good starting point is just a simple linear regression. We take the state, the action and the output needs to be the ending state.


A reasonable model for this would be a neural network with no activation function and two outputs. For the x and y of this.

I think the action could be better represented in a one hot encoding format than in a numerical format.

For the inverse dynamics model, the output could be coupled with a sigmoid function and a softmax function to produce probabilities of things.

The forward dynamics model is primarily a regression model and the inverse dynamics model is primarily a classification based neural network.

In [6]:
class ForwardDynamicsModel(nn.Module):
    def __init__(self):
        super(ForwardDynamicsModel, self).__init__()
        # 3 inputs -> 2 outputs
        self.forward_model = nn.Sequential(
            nn.Linear(6, 100),  # Input layer to hidden layer
            nn.Sigmoid(),  # Sigmoid activation
            nn.Linear(100, 100),  # Input layer to hidden layer
            nn.Sigmoid(),  # Sigmoid activation
            nn.Linear(100, 2),  # Hidden layer to output layer
        )

    def forward(self, x):
        return self.forward_model(x)

In [7]:
model_forward = ForwardDynamicsModel()
criterion_forward = nn.MSELoss()
optimizer_forward = torch.optim.SGD(model_forward.parameters(), lr=0.000001)




In [ ]:
# Load the state dict
# model.load_state_dict(torch.load("forward_model.pth"))


In [8]:
# Training the model separately
# for i in tqdm(range(20000)):

#     data_X, data_Y = do_n_samples(2000)

#     x = torch.tensor(data_X).float()
#     y = torch.tensor(data_Y).float()

#     predicted_output = model_forward(x)
#     loss = criterion_forward(y, predicted_output)
#     if i%500 == 0:
#         print(loss)
#     loss.backward()
#     optimizer_forward.step()

  0%|          | 8/20000 [00:00<08:31, 39.10it/s]

tensor(13.1249, grad_fn=<MseLossBackward0>)


  3%|▎         | 502/20000 [00:08<04:58, 65.31it/s]

tensor(12.4098, grad_fn=<MseLossBackward0>)


  5%|▌         | 1010/20000 [00:17<05:14, 60.44it/s]

tensor(7.3980, grad_fn=<MseLossBackward0>)


... (71 outputs left out)

tensor(1.4470, grad_fn=<MseLossBackward0>)


 98%|█████████▊| 19509/20000 [05:42<00:08, 54.70it/s]

tensor(1.2353, grad_fn=<MseLossBackward0>)


100%|██████████| 20000/20000 [05:51<00:00, 56.94it/s]


In [12]:
# torch.save(model_forward.state_dict(), "forward_model_without_intrinsic.pth")

In [13]:
class InverseDynamicsModel(nn.Module):
    def __init__(self):
        super(InverseDynamicsModel, self).__init__()
        # No hidden layer: direct linear from 4 inputs to 4 outputs
        self.inverse_model = nn.Sequential(
            nn.Linear(4, 100),  # Input layer to hidden layer
            nn.ReLU(),  # Sigmoid activation
            nn.Linear(100, 100),  # Input layer to hidden layer
            nn.ReLU(),  # Sigmoid activation
            nn.Linear(100, 4),  # Hidden layer to output layer
        )

    def forward(self, x):
        x = self.inverse_model(x)  # Linear transformation
        x = F.softmax(x, dim=1)  # Softmax over the 4 outputs (per sample)
        return x

We also need to add intrinsic motivation to the mix. Essentially, intrinsic motivation would be the loss of the forward dynamics model. Essentially, it would be the difference in prediction for the next state and the actual next state. This would be multiplied by eta to compute intrinsic motivation.

In [14]:
model_inverse = InverseDynamicsModel()
criterion_inverse = nn.CrossEntropyLoss()
optimizer_inverse = torch.optim.SGD(model_inverse.parameters(), lr=0.000001)



In [15]:
# Training the inverse dynamics model separately

# for i in tqdm(range(30000)):

#     data_X, data_Y = do_n_samples(2000, inverse=True)

#     x = torch.tensor(data_X).float()
#     y = torch.tensor(data_Y).float()

#     predicted_output = model_inverse(x)
#     loss = criterion_inverse(y, predicted_output)
#     if i%500 == 0:
#         print(loss)
#     loss.backward()
#     optimizer_inverse.step()

  0%|          | 7/30000 [00:00<08:01, 62.28it/s]

tensor(1.4930, grad_fn=<DivBackward1>)


  2%|▏         | 516/30000 [00:09<08:20, 58.92it/s]

tensor(1.4950, grad_fn=<DivBackward1>)


  3%|▎         | 1012/30000 [00:18<08:10, 59.13it/s]

tensor(1.4867, grad_fn=<DivBackward1>)


... (111 outputs left out)

tensor(0.8618, grad_fn=<DivBackward1>)


 98%|█████████▊| 29510/30000 [09:07<00:09, 50.82it/s]

tensor(0.8397, grad_fn=<DivBackward1>)


100%|██████████| 30000/30000 [09:17<00:00, 53.86it/s]


In [16]:
# torch.save(model_inverse.state_dict(), "inverse_model_without_intrinsic.pth")

Basically, we want to optimize things in a way that would find the parameters for lowering this particular loss.

The loss function would occur as follows:

-lambda * intrinsic_motivation_reward + (1-beta) * inverse_model_loss + (beta) * forward_model_loss

We would like to find the parameters that would minimize this loss.

Henceforth, the backpropogation must also happen in a similar way. It needs to combine the individual losses and intrinsic motivation and aim to find the derivatives that would minimize that loss function.

For starters we can try to train them separately and thenn combine them in some way.

What would the policy look like?

The policy should look to maximize intrinsic motivation.

The policy will be a NN similar to the inverse dynamics model. It will take in the current state and output the action. It will have sigmoidal function coupled with a softmax function..

In [10]:
class PolicyNetwork(nn.Module):
    def __init__(self):
        super(PolicyNetwork, self).__init__()
        # No hidden layer: direct linear from 4 inputs to 4 outputs
        self.policy_network_stack = nn.Sequential(
            nn.Linear(2, 100),  # Input layer to hidden layer
            nn.ReLU(),  # Sigmoid activation
            nn.Linear(100, 100),  # Input layer to hidden layer
            nn.ReLU(),  # Sigmoid activation
            nn.Linear(100, 4),  # Hidden layer to output layer
        )

    def forward(self, x):
        x = self.policy_network_stack(x)  # Linear transformation
        # print(x)
        x = F.softmax(x, dim=1)  # Softmax over the 4 outputs (per sample)
        return x

The policy network's aim would be to maximize intrinsic motivation. Hence, it would be perhaps a gradient ascent style thing where the derivatives computed would be about the intrinsic motivation and trying to maximize that.

Let's start with something simple and train the forward dynamics model with a random policy to generate the data just for some practice and then we can scale things up from that.

In [16]:
policy = PolicyNetwork()

criterion_policy = nn.CrossEntropyLoss()

policy

PolicyNetwork(
  (policy_network_stack): Sequential(
    (0): Linear(in_features=2, out_features=100, bias=True)
    (1): ReLU()
    (2): Linear(in_features=100, out_features=100, bias=True)
    (3): ReLU()
    (4): Linear(in_features=100, out_features=4, bias=True)
  )
)

In [15]:
combined_optimizer = torch.optim.Adam(
    list(model_forward.parameters()) + list(model_inverse.parameters()) + list(policy.parameters()),
    lr=0.000001
)

In [12]:
def get_n_samples_intrinsic_motivation(n):

  data_X_forward = []
  data_Y_forward = []

  data_X_inverse = []
  data_Y_inverse = []

  for i in range(n):

    previous_state = env.agent_location_x, env.agent_location_y
    previous_state_tensor = torch.tensor([previous_state]).float()

    probs = policy(previous_state_tensor)[0]


    action = torch.multinomial(probs, num_samples=1).item()
    next_state = env.step(action)


    input = list(previous_state) + one_hot_encoding(action)
    output = list(next_state)

    data_X_forward.append(input)
    data_Y_forward.append(output)



    input = list(previous_state) + list(next_state)
    output = one_hot_encoding(action)
    data_X_inverse.append(input)
    data_Y_inverse.append(output)

  return data_X_forward, data_Y_forward, data_X_inverse, data_Y_inverse


Here we have essentially used the policy network as a probability distribution where the policy network outputs the probabilities upon which the action is selected based upon that probability distirbution

In [18]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [20]:

with open('data_Y_forward.csv', 'a', newline='') as file:
  writer = csv.writer(file)


  for epoch in tqdm(range(30000)):


    data_X_forward, data_Y_forward, data_X_inverse, data_Y_inverse = get_n_samples_intrinsic_motivation(2000)

    writer.writerow(data_Y_forward)

    # all_data_Y_forward.extend(data_Y_forward
    # all_data_X_forward.extend(data_X_forward)

    x_forward = torch.tensor(data_X_forward).float()
    y_forward = torch.tensor(data_Y_forward).float()


    x_inverse = torch.tensor(data_X_inverse).float()
    y_inverse = torch.tensor(data_Y_inverse).float()

    forward_output = model_forward(x_forward)
    forward_loss = criterion_forward(y_forward, forward_output)

    inverse_output = model_inverse(x_inverse)
    inverse_loss = criterion_inverse(y_inverse, inverse_output)


    eta = 3
    intrinsic_motivation_reward = eta * forward_loss

    beta = 0.1
    intrinsic_lambda = 1.2
    combined_loss = -intrinsic_lambda * intrinsic_motivation_reward + (1-beta) * inverse_loss + (beta) * forward_loss


    if epoch % 100 == 0:
      print()
      print(forward_loss)
      print(inverse_loss)
      print(combined_loss)


    combined_optimizer.zero_grad()
    combined_loss.backward()
    combined_optimizer.step()

    # optimizer_forward.zero_grad()
    # optimizer_inverse.zero_grad()

    # combined_loss.backward()
    # optimizer_forward.step()
    # optimizer_inverse.step()

  0%|          | 1/30000 [00:00<4:26:09,  1.88it/s]


tensor(25.1925, grad_fn=<MseLossBackward0>)
tensor(1.4903, grad_fn=<DivBackward1>)
tensor(-86.8324, grad_fn=<AddBackward0>)


  0%|          | 101/30000 [00:48<4:11:47,  1.98it/s]


tensor(16.9418, grad_fn=<MseLossBackward0>)
tensor(1.4953, grad_fn=<DivBackward1>)
tensor(-57.9505, grad_fn=<AddBackward0>)


  1%|          | 201/30000 [01:35<4:00:37,  2.06it/s]


tensor(23.5099, grad_fn=<MseLossBackward0>)
tensor(1.4869, grad_fn=<DivBackward1>)
tensor(-80.9464, grad_fn=<AddBackward0>)


... (591 outputs left out)


tensor(65.8330, grad_fn=<MseLossBackward0>)
tensor(1.2538, grad_fn=<DivBackward1>)
tensor(-229.2871, grad_fn=<AddBackward0>)


100%|█████████▉| 29901/30000 [3:56:47<00:46,  2.14it/s]


tensor(46.5733, grad_fn=<MseLossBackward0>)
tensor(1.2745, grad_fn=<DivBackward1>)
tensor(-161.8594, grad_fn=<AddBackward0>)


100%|██████████| 30000/30000 [3:57:35<00:00,  2.10it/s]


In [21]:
torch.save(model_inverse.state_dict(), "inverse_model.pth")
torch.save(model_forward.state_dict(), "forward_model.pth")
torch.save(policy.state_dict(), "policy_model.pth")

There are two forms of data that we need to collect.
1. You want to collect the data for previous state, action -> next state
This is the forward dynamics model.
2. You want to collect the data for previous state, next state -> action
This is the inverse dynamics model.

Now, you also want to train the policy network.

The policy network would be basically be trained by the combined losses of forward, inverse and intrinsic motivaiton. Intrinsic motivation in this case would be loss of the forward dynamics model multiplied by eta.


You would use the data above to compute individual forward and inverse losses. Then, you would use the forward loss to compute intrinsic motivaiton.

You would then compute a combined loss.

Once, the combined loss is computed, this loss would then be used to train the three networks. Forward, inverse and policy

Here is how to think about the intrinsic motivation loss.

We are going to use the policy network to output the action based upon the large policy network and have it output actions based upon all that.

Then. use those actions to produce data and then compute the losses in the forward dynamics model and the inverse dynamics model. The forward dynamics model would essentially be the case that we are trying to determine how accurately can the agent detect the next state based upon the previous state and action. The inverse dynamics model is essentially given a previous state and next state, how accurately can you determine the action.

Intrinsic motivation is the eta times the loss of the forward dynamics model.